In [2]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error,r2_score
import numpy as np

In [3]:
data = load_diabetes()
X,y = data.data,data.target
print("X shape",X.shape)


X shape (442, 10)


In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [5]:
model = DecisionTreeRegressor(max_depth=3)
model.fit(X_train,y_train)



DecisionTreeRegressor(max_depth=3)

In [6]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test,y_pred))
r2 = r2_score(y_test,y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R^2 : {r2:.3f}")

RMSE: 59.60
R^2 : 0.329


In [7]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/mirichoi0218/insurance/insurance.csv")
df.head()

y = df['charges']
X = df.drop(columns=['charges'])
X = pd.get_dummies(X, columns=['sex','smoker','region'],drop_first=True)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = DecisionTreeRegressor(max_depth=4)
model.fit(X_train,y_train)

DecisionTreeRegressor(max_depth=4)

In [8]:
y_pred = model.predict(X_test)

import numpy as np

from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:,.2f}")    
print(f"MAE : {mae:,.2f}")
print(f"R^2 : {r2:.3f}")    

RMSE: 4,592.76
MAE : 2,697.77
R^2 : 0.864


**HOW TO FIND GOOD HYPERPARAMETERS**


In [9]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# Sweep max_depth, record train AND test R^2
for d in range(1, 16):
    t = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    train_r2 = r2_score(y_train, t.predict(X_train))   # how well it fits TRAINING data
    test_r2  = r2_score(y_test,  t.predict(X_test))     # how well it GENERALIZES
    print(f"depth {d:>2}: train R2 {train_r2:.3f} | test R2 {test_r2:.3f}")

depth  1: train R2 0.608 | test R2 0.660
depth  2: train R2 0.823 | test R2 0.832
depth  3: train R2 0.854 | test R2 0.853
depth  4: train R2 0.865 | test R2 0.864
depth  5: train R2 0.880 | test R2 0.834
depth  6: train R2 0.891 | test R2 0.806
depth  7: train R2 0.911 | test R2 0.801
depth  8: train R2 0.933 | test R2 0.799
depth  9: train R2 0.949 | test R2 0.770
depth 10: train R2 0.965 | test R2 0.750
depth 11: train R2 0.981 | test R2 0.679
depth 12: train R2 0.991 | test R2 0.722
depth 13: train R2 0.994 | test R2 0.746
depth 14: train R2 0.997 | test R2 0.702
depth 15: train R2 0.998 | test R2 0.706


In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

# 1. Define the grid of values to try (can tune several params at once)
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

# 2. GridSearchCV trains a model for EVERY combo, using 5-fold CV on the TRAIN set
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,                       # 5-fold cross-validation
    scoring='r2',               # what to optimize (use 'accuracy'/'f1' for classification)
    n_jobs=-1)                  # use all CPU cores
grid.fit(X_train, y_train)      # ONLY training data goes in

# 3. Read off the best settings (chosen by CV, not by the test set)
print("Best params:", grid.best_params_)
print("Best CV R^2:", round(grid.best_score_, 3))

# 4. NOW evaluate the winner once on the untouched test set
best = grid.best_estimator_
print("Test R^2:", round(best.score(X_test, y_test), 3))

Best params: {'max_depth': 4, 'min_samples_leaf': 1}
Best CV R^2: 0.837
Test R^2: 0.864


**tree cleanup first (pruning + feature_importances_ + entropy)**